# Transformer from Scratch

Full implementation: MultiHeadAttention, FeedForward, Encoder, Decoder, and the complete Transformer.

**Interview questions this covers:**
- "Walk me through the transformer architecture."
- "Pre-norm vs post-norm — which is better and why?"
- "Explain the masking strategies."

---

## Key Formulas

**Scaled dot-product attention:**
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

**Multi-head attention:**
$$\text{MultiHead}(Q,K,V) = \text{Concat}(\text{head}_1, \dots, \text{head}_h)W^O$$
$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Parameter count (per layer):**
- MHA: $4 d_{\text{model}}^2$ (Q, K, V, O projections)
- FFN: $2 \cdot d_{\text{model}} \cdot d_{\text{ff}} = 2 \cdot d_{\text{model}} \cdot 4d_{\text{model}} = 8d_{\text{model}}^2$
- LayerNorms: $4 d_{\text{model}}$ (2 norms, each with scale + bias)
- **Total per encoder layer:** $\approx 12 d_{\text{model}}^2$

**Memory complexity:** $O(n^2 \cdot d)$ for attention (sequence length $n$, head dim $d$)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Scaled Dot-Product Attention

The core primitive. Everything else is built on top of this.

In [ ]:
def scaled_dot_product_attention(
    query: torch.Tensor,   # (batch, heads, seq_q, d_k)
    key: torch.Tensor,     # (batch, heads, seq_k, d_k)
    value: torch.Tensor,   # (batch, heads, seq_k, d_v)
    mask: torch.Tensor = None,
    dropout: nn.Dropout = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Returns (output, attention_weights)."""
    d_k = query.size(-1)
    # (batch, heads, seq_q, seq_k)
    scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
    
    if mask is not None:
        # mask: True where we want to BLOCK attention (convention: True = masked out)
        scores = scores.masked_fill(mask, float('-inf'))
    
    attn_weights = F.softmax(scores, dim=-1)
    
    if dropout is not None:
        attn_weights = dropout(attn_weights)
    
    output = torch.matmul(attn_weights, value)  # (batch, heads, seq_q, d_v)
    return output, attn_weights

# Quick test
B, H, S, D = 2, 4, 8, 64
q = torch.randn(B, H, S, D)
k = torch.randn(B, H, S, D)
v = torch.randn(B, H, S, D)
out, weights = scaled_dot_product_attention(q, k, v)
print(f"Output shape: {out.shape}")   # (2, 4, 8, 64)
print(f"Weights shape: {weights.shape}")  # (2, 4, 8, 8)
print(f"Weights sum to 1: {weights.sum(-1)[0, 0]}")  # should be all 1s

## 2. Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """Standard multi-head attention.
    
    Design choices:
    - Single large linear for Q/K/V then reshape, vs separate linears per head.
      Single large linear is standard (more efficient, same expressiveness).
    - Output projection after concatenation.
    """
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads  # head dimension
        
        # Combined Q, K, V projection (could also be three separate linears)
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
    
    def forward(
        self,
        query: torch.Tensor,    # (batch, seq_q, d_model)
        key: torch.Tensor,      # (batch, seq_k, d_model)
        value: torch.Tensor,    # (batch, seq_k, d_model)
        mask: torch.Tensor = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        batch_size = query.size(0)
        
        # Project and reshape: (batch, seq, d_model) -> (batch, n_heads, seq, d_k)
        Q = self.W_q(query).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(key).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(value).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        
        # Attention
        attn_output, attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )
        
        # Concatenate heads: (batch, n_heads, seq_q, d_k) -> (batch, seq_q, d_model)
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        return self.W_o(attn_output), attn_weights

# Test
mha = MultiHeadAttention(d_model=256, n_heads=8)
x = torch.randn(2, 10, 256)
out, weights = mha(x, x, x)  # self-attention
print(f"MHA output: {out.shape}")      # (2, 10, 256)
print(f"MHA weights: {weights.shape}")  # (2, 8, 10, 10)
print(f"MHA params: {sum(p.numel() for p in mha.parameters()):,}")  # 4 * 256^2 + biases

## 3. Position-wise Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    """Position-wise FFN: two linear layers with activation in between.
    
    Standard: ReLU (original paper), modern: GELU or SwiGLU (LLaMA, PaLM).
    d_ff is typically 4 * d_model (original) or 8/3 * d_model with SwiGLU.
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1, 
                 activation: str = 'relu'):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'gelu':
            self.activation = nn.GELU()
        else:
            raise ValueError(f"Unknown activation: {activation}")
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(self.activation(self.linear1(x))))


class SwiGLUFeedForward(nn.Module):
    """SwiGLU FFN used in LLaMA, PaLM.
    
    SwiGLU(x) = (xW_1 * Swish(xW_gate)) W_2
    Uses 3 weight matrices instead of 2, so d_ff is reduced to ~(2/3)*4*d_model
    to keep param count similar.
    """
    def __init__(self, d_model: int, d_ff: int = None, dropout: float = 0.1):
        super().__init__()
        d_ff = d_ff or int(4 * d_model * 2 / 3)
        # Round to nearest multiple of 64 for hardware efficiency
        d_ff = ((d_ff + 63) // 64) * 64
        
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w_gate = nn.Linear(d_model, d_ff, bias=False)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.w2(F.silu(self.w_gate(x)) * self.w1(x)))

# Test both
ffn_relu = FeedForward(256)
ffn_swiglu = SwiGLUFeedForward(256)
x = torch.randn(2, 10, 256)
print(f"ReLU FFN params: {sum(p.numel() for p in ffn_relu.parameters()):,}")
print(f"SwiGLU FFN params: {sum(p.numel() for p in ffn_swiglu.parameters()):,}")
print(f"ReLU FFN output: {ffn_relu(x).shape}")
print(f"SwiGLU FFN output: {ffn_swiglu(x).shape}")

## 4. Masking Strategies

Three types of masks:
1. **Padding mask**: ignore pad tokens in attention
2. **Causal mask**: prevent attending to future positions (decoder)
3. **Cross-attention mask**: encoder-decoder, padding mask on encoder outputs

In [ ]:
def create_padding_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """Create mask where True = position to mask out.
    Shape: (batch, 1, 1, seq_len) — broadcasts over heads and query positions.
    """
    return (seq == pad_idx).unsqueeze(1).unsqueeze(2)  # (B, 1, 1, S)


def create_causal_mask(seq_len: int, device=None) -> torch.Tensor:
    """Upper-triangular mask: True above diagonal = masked out.
    Shape: (1, 1, seq_len, seq_len) — broadcasts over batch and heads.
    """
    mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1).bool()
    return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, S, S)


def create_combined_mask(seq: torch.Tensor, pad_idx: int = 0) -> torch.Tensor:
    """Combine padding + causal mask for decoder self-attention."""
    pad_mask = create_padding_mask(seq, pad_idx)  # (B, 1, 1, S)
    causal_mask = create_causal_mask(seq.size(1), device=seq.device)  # (1, 1, S, S)
    return pad_mask | causal_mask  # broadcast: (B, 1, S, S)

# Visualize masks
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Padding mask
seq = torch.tensor([[5, 3, 7, 0, 0]])  # 0 = pad
pad_mask = create_padding_mask(seq)
axes[0].imshow(pad_mask[0, 0].expand(5, -1).float(), cmap='Reds', vmin=0, vmax=1)
axes[0].set_title('Padding Mask')
axes[0].set_xlabel('Key position')
axes[0].set_ylabel('Query position')

# Causal mask
causal = create_causal_mask(5)
axes[1].imshow(causal[0, 0].float(), cmap='Reds', vmin=0, vmax=1)
axes[1].set_title('Causal Mask')
axes[1].set_xlabel('Key position')

# Combined
combined = create_combined_mask(seq)
axes[2].imshow(combined[0, 0].float(), cmap='Reds', vmin=0, vmax=1)
axes[2].set_title('Combined (Pad + Causal)')
axes[2].set_xlabel('Key position')

plt.suptitle('Red = masked out (cannot attend)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Transformer Encoder Layer

**Post-norm (original):** `x = LayerNorm(x + Sublayer(x))`

**Pre-norm (GPT-2, LLaMA, modern):** `x = x + Sublayer(LayerNorm(x))`

Pre-norm is now preferred because:
- More stable training (gradients flow through residual stream unimpeded)
- Often no learning rate warmup needed
- Slightly worse final performance at small scale, but enables training much deeper models

In [ ]:
class TransformerEncoderLayer(nn.Module):
    """Single encoder layer with configurable pre-norm or post-norm."""
    def __init__(self, d_model: int, n_heads: int, d_ff: int = None,
                 dropout: float = 0.1, pre_norm: bool = True):
        super().__init__()
        self.pre_norm = pre_norm
        
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x: torch.Tensor, src_mask: torch.Tensor = None) -> torch.Tensor:
        if self.pre_norm:
            # Pre-norm: norm before sublayer
            normed = self.norm1(x)
            attn_out, _ = self.self_attn(normed, normed, normed, mask=src_mask)
            x = x + self.dropout1(attn_out)
            
            normed = self.norm2(x)
            x = x + self.dropout2(self.ffn(normed))
        else:
            # Post-norm: norm after sublayer (original Vaswani)
            attn_out, _ = self.self_attn(x, x, x, mask=src_mask)
            x = self.norm1(x + self.dropout1(attn_out))
            
            x = self.norm2(x + self.dropout2(self.ffn(x)))
        
        return x

# Test
enc_layer = TransformerEncoderLayer(d_model=256, n_heads=8, pre_norm=True)
x = torch.randn(2, 10, 256)
out = enc_layer(x)
print(f"Encoder layer output: {out.shape}")
print(f"Encoder layer params: {sum(p.numel() for p in enc_layer.parameters()):,}")

## 6. Transformer Decoder Layer

Three sub-layers:
1. Masked self-attention (causal)
2. Cross-attention (queries from decoder, keys/values from encoder)
3. Feed-forward

In [ ]:
class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int = None,
                 dropout: float = 0.1, pre_norm: bool = True):
        super().__init__()
        self.pre_norm = pre_norm
        
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.ffn = FeedForward(d_model, d_ff, dropout)
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(
        self,
        x: torch.Tensor,                    # decoder input
        encoder_output: torch.Tensor,        # encoder output (memory)
        tgt_mask: torch.Tensor = None,       # causal + padding for decoder
        memory_mask: torch.Tensor = None,    # padding mask on encoder outputs
    ) -> torch.Tensor:
        if self.pre_norm:
            # 1. Masked self-attention
            normed = self.norm1(x)
            attn_out, _ = self.self_attn(normed, normed, normed, mask=tgt_mask)
            x = x + self.dropout1(attn_out)
            
            # 2. Cross-attention
            normed = self.norm2(x)
            attn_out, _ = self.cross_attn(normed, encoder_output, encoder_output,
                                           mask=memory_mask)
            x = x + self.dropout2(attn_out)
            
            # 3. FFN
            normed = self.norm3(x)
            x = x + self.dropout3(self.ffn(normed))
        else:
            attn_out, _ = self.self_attn(x, x, x, mask=tgt_mask)
            x = self.norm1(x + self.dropout1(attn_out))
            
            attn_out, _ = self.cross_attn(x, encoder_output, encoder_output,
                                           mask=memory_mask)
            x = self.norm2(x + self.dropout2(attn_out))
            
            x = self.norm3(x + self.dropout3(self.ffn(x)))
        
        return x

# Test
dec_layer = TransformerDecoderLayer(d_model=256, n_heads=8)
tgt = torch.randn(2, 8, 256)
memory = torch.randn(2, 12, 256)
causal = create_causal_mask(8)
out = dec_layer(tgt, memory, tgt_mask=causal)
print(f"Decoder layer output: {out.shape}")
print(f"Decoder layer params: {sum(p.numel() for p in dec_layer.parameters()):,}")

## 7. Sinusoidal Positional Encoding

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{\text{model}}}}\right)$$

In [ ]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (batch, seq_len, d_model)"""
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

# Quick visualization
pe = SinusoidalPositionalEncoding(64, dropout=0.0)
plt.figure(figsize=(10, 4))
plt.imshow(pe.pe[0, :50, :].numpy(), aspect='auto', cmap='RdBu')
plt.xlabel('Dimension')
plt.ylabel('Position')
plt.title('Sinusoidal Positional Encoding')
plt.colorbar()
plt.tight_layout()
plt.show()

## 8. Full Transformer (Encoder-Decoder)

In [ ]:
class Transformer(nn.Module):
    """Full encoder-decoder transformer.
    
    Architecture notes:
    - Pre-norm with final layer norm (important: without this, pre-norm output is unnormalized)
    - Tied embeddings between encoder/decoder and output projection (optional, saves params)
    - Scaling: multiply embeddings by sqrt(d_model) before adding positional encoding
    """
    def __init__(
        self,
        src_vocab_size: int,
        tgt_vocab_size: int,
        d_model: int = 512,
        n_heads: int = 8,
        n_encoder_layers: int = 6,
        n_decoder_layers: int = 6,
        d_ff: int = 2048,
        dropout: float = 0.1,
        max_len: int = 5000,
        pre_norm: bool = True,
        tie_embeddings: bool = False,
    ):
        super().__init__()
        self.d_model = d_model
        self.pre_norm = pre_norm
        
        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len, dropout)
        
        # Encoder
        self.encoder_layers = nn.ModuleList([
            TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, pre_norm)
            for _ in range(n_encoder_layers)
        ])
        
        # Decoder
        self.decoder_layers = nn.ModuleList([
            TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, pre_norm)
            for _ in range(n_decoder_layers)
        ])
        
        # Final layer norm (needed for pre-norm)
        if pre_norm:
            self.encoder_norm = nn.LayerNorm(d_model)
            self.decoder_norm = nn.LayerNorm(d_model)
        
        # Output projection
        self.output_proj = nn.Linear(d_model, tgt_vocab_size)
        
        if tie_embeddings:
            self.output_proj.weight = self.tgt_embedding.weight
        
        self._init_weights()
    
    def _init_weights(self):
        """Xavier uniform initialization (standard for transformers)."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def encode(self, src: torch.Tensor, src_mask: torch.Tensor = None) -> torch.Tensor:
        x = self.pos_encoding(self.src_embedding(src) * math.sqrt(self.d_model))
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        if self.pre_norm:
            x = self.encoder_norm(x)
        return x
    
    def decode(
        self,
        tgt: torch.Tensor,
        memory: torch.Tensor,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        x = self.pos_encoding(self.tgt_embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.decoder_layers:
            x = layer(x, memory, tgt_mask, memory_mask)
        if self.pre_norm:
            x = self.decoder_norm(x)
        return x
    
    def forward(
        self,
        src: torch.Tensor,
        tgt: torch.Tensor,
        src_mask: torch.Tensor = None,
        tgt_mask: torch.Tensor = None,
        memory_mask: torch.Tensor = None,
    ) -> torch.Tensor:
        memory = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, memory, tgt_mask, memory_mask)
        return self.output_proj(decoder_output)

# Test
model = Transformer(
    src_vocab_size=1000, tgt_vocab_size=1000,
    d_model=128, n_heads=4, n_encoder_layers=2, n_decoder_layers=2,
    d_ff=512, dropout=0.1
)
src = torch.randint(0, 1000, (2, 10))
tgt = torch.randint(0, 1000, (2, 8))
tgt_mask = create_causal_mask(8)
logits = model(src, tgt, tgt_mask=tgt_mask)
print(f"Output logits: {logits.shape}")  # (2, 8, 1000)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 9. Demo: Copy Task

Train the transformer to copy input sequences. This is the simplest possible seq2seq task — if the model can't learn this, something is broken.

In [ ]:
# Copy task: input [BOS, x1, x2, ..., xn, EOS] -> output [BOS, x1, x2, ..., xn, EOS]
VOCAB_SIZE = 20
PAD_IDX = 0
BOS_IDX = 1
EOS_IDX = 2
SEQ_LEN = 8  # content tokens (excluding BOS/EOS)

def generate_copy_batch(batch_size: int, seq_len: int = SEQ_LEN):
    """Generate a batch of copy-task data."""
    # Random tokens from [3, VOCAB_SIZE) — avoid special tokens
    content = torch.randint(3, VOCAB_SIZE, (batch_size, seq_len))
    
    # Source: [BOS, content, EOS]
    src = torch.cat([
        torch.full((batch_size, 1), BOS_IDX),
        content,
        torch.full((batch_size, 1), EOS_IDX),
    ], dim=1)
    
    # Target input: [BOS, content] (teacher forcing)
    tgt_input = torch.cat([
        torch.full((batch_size, 1), BOS_IDX),
        content,
    ], dim=1)
    
    # Target output: [content, EOS] (what we predict)
    tgt_output = torch.cat([
        content,
        torch.full((batch_size, 1), EOS_IDX),
    ], dim=1)
    
    return src, tgt_input, tgt_output

src, tgt_in, tgt_out = generate_copy_batch(4)
print(f"Source:      {src[0].tolist()}")
print(f"Target in:   {tgt_in[0].tolist()}")
print(f"Target out:  {tgt_out[0].tolist()}")

In [ ]:
# Train the copy task
model = Transformer(
    src_vocab_size=VOCAB_SIZE, tgt_vocab_size=VOCAB_SIZE,
    d_model=64, n_heads=4, n_encoder_layers=2, n_decoder_layers=2,
    d_ff=128, dropout=0.1, pre_norm=True
)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

losses = []
model.train()

for step in range(500):
    src, tgt_in, tgt_out = generate_copy_batch(32)
    src, tgt_in, tgt_out = src.to(device), tgt_in.to(device), tgt_out.to(device)
    
    tgt_mask = create_causal_mask(tgt_in.size(1), device=device)
    
    logits = model(src, tgt_in, tgt_mask=tgt_mask)
    loss = criterion(logits.reshape(-1, VOCAB_SIZE), tgt_out.reshape(-1))
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    
    losses.append(loss.item())
    if (step + 1) % 100 == 0:
        print(f"Step {step+1}, Loss: {loss.item():.4f}")

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Copy Task Training Loss')
plt.tight_layout()
plt.show()

In [ ]:
# Greedy decode to test
@torch.no_grad()
def greedy_decode(model, src, max_len=12):
    model.eval()
    memory = model.encode(src)
    
    # Start with BOS
    ys = torch.full((src.size(0), 1), BOS_IDX, dtype=torch.long, device=src.device)
    
    for _ in range(max_len):
        tgt_mask = create_causal_mask(ys.size(1), device=src.device)
        logits = model.output_proj(model.decode(ys, memory, tgt_mask=tgt_mask))
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)
        ys = torch.cat([ys, next_token], dim=1)
        if (next_token == EOS_IDX).all():
            break
    
    return ys

# Test on fresh data
src_test, _, tgt_test = generate_copy_batch(5)
src_test = src_test.to(device)
predictions = greedy_decode(model, src_test)

print("Source -> Predicted (expected = source content)")
for i in range(5):
    print(f"  {src_test[i].tolist()} -> {predictions[i].tolist()}")

## 10. Parameter Count Analysis

In [ ]:
def count_parameters(model):
    """Detailed parameter count breakdown."""
    total = 0
    breakdown = {}
    for name, param in model.named_parameters():
        n = param.numel()
        total += n
        # Group by component
        component = name.split('.')[0]
        breakdown[component] = breakdown.get(component, 0) + n
    
    print(f"{'Component':<25} {'Parameters':>12} {'Percentage':>10}")
    print('-' * 50)
    for comp, count in sorted(breakdown.items(), key=lambda x: -x[1]):
        print(f"{comp:<25} {count:>12,} {count/total*100:>9.1f}%")
    print('-' * 50)
    print(f"{'TOTAL':<25} {total:>12,}")
    return total

# Analyze our model
print("=== Copy Task Model ===")
count_parameters(model)

print("\n=== GPT-2 Scale (d=768, 12 layers, 12 heads) ===")
gpt2_scale = Transformer(
    src_vocab_size=50257, tgt_vocab_size=50257,
    d_model=768, n_heads=12, n_encoder_layers=12, n_decoder_layers=12,
    d_ff=3072
)
count_parameters(gpt2_scale)

## Interview Notes

**"Walk me through the transformer architecture."**
- Encoder: N layers of (self-attention + FFN), each with residual connections and layer norm
- Decoder: N layers of (masked self-attention + cross-attention + FFN)
- Input: token embeddings + positional encoding, scaled by sqrt(d_model)
- Output: linear projection to vocabulary size + softmax

**"Pre-norm vs post-norm?"**
- Post-norm (original): `LayerNorm(x + Sublayer(x))` — norm after residual add
- Pre-norm (modern): `x + Sublayer(LayerNorm(x))` — norm before sublayer
- Pre-norm: gradients flow through residual stream without being normalized, more stable for deep models
- Post-norm: slightly better final performance at small scale, but harder to train deep models
- Nearly all modern LLMs use pre-norm (GPT-2+, LLaMA, etc.)
- Pre-norm with RMSNorm (no centering, just scaling) is the current standard (LLaMA)

**"Why scale by sqrt(d_k)?"**
- Dot products grow with dimension: if q, k ~ N(0,1), then q·k ~ N(0, d_k)
- Large values push softmax into saturation (tiny gradients)
- Dividing by sqrt(d_k) keeps variance at 1

**"Why scale embeddings by sqrt(d_model)?"**
- Embedding vectors have small magnitudes (initialized near zero)
- Positional encodings have magnitude ~1
- Scaling ensures embeddings and positional encodings are on similar scale